# GLiNER2 Synthetic Data Generator Demo

This notebook demonstrates the `DataGenerator` class from `generate.py`, which turns a natural language task description into synthetic GLiNER2-style training examples.

By default, the generator infers tasks using lightweight rules, but you can also enable an optional LLM-based inference mode backed by a local Ollama `llama3.2` model.

It also includes a scaffold for fine-tuning `fastino/gliner2-base-v1` on the generated data using the GLiNER2 training API.

In [ ]:
from pprint import pprint

from generate import DataGenerator

generator = DataGenerator(seed=42)

## Single-task examples

We first generate examples for each of the four GLiNER2 task types:

- NER
- Classification
- Relation extraction
- JSON extraction


In [ ]:
single_task_descriptions = [
    "Extract company, person and location names from short tech news snippets.",  # NER
    "Classify customer review sentiment into positive, negative and neutral.",    # Classification
    "Identify works_for and lives_in relations between people and organizations.", # Relation extraction
    "Extract product name, storage and price into a JSON structure.",             # JSON extraction
]

for desc in single_task_descriptions:
    print("==== Description ====")
    print(desc)
    examples = generator.generate(desc, n=3)
    print("First example:")
    pprint(examples[0])
    print()

## Multi-task examples

Now we use composite task descriptions that implicitly require multiple task types to be active at once.


In [ ]:
multi_task_descriptions = [
    # NER + classification
    "From each customer review, extract company names and classify overall sentiment into positive, negative or neutral.",
    # JSON extraction + classification
    "Extract a JSON product record (name, storage, price) from each sentence and classify whether the tone is positive, negative or neutral.",
]

for desc in multi_task_descriptions:
    print("==== Multi-task description ====")
    print(desc)
    examples = generator.generate(desc, n=4)
    print("Example 0:")
    pprint(examples[0])
    print("Example 1:")
    pprint(examples[1])
    print()

## Optional: LLM-backed task inference (Ollama `llama3.2`)

If you have Ollama running locally with a `llama3.2` model available, you can enable **LLM-based task inference**.

This section:
- checks whether the local Ollama server is reachable
- compares rule-based vs LLM-inferred task settings
- runs `generate()` with `task_inference_mode="llm"` to confirm end-to-end compatibility

In [ ]:
import json
import urllib.error
import urllib.request
from dataclasses import asdict

from generate import DataGenerator

import os
import requests


def _http_get_json(url: str, timeout_s: int = 3) -> dict | None:
    try:
        with urllib.request.urlopen(url, timeout=timeout_s) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError):
        return None


def ollama_status(base_url: str | None = None) -> dict:
    """Return basic reachability + available model names for the Ollama server.

    If base_url is not provided, this uses the OLLAMA_HOST environment variable
    when set (e.g. '172.21.112.1:11434' or 'http://172.21.112.1:11434'), and
    falls back to 'http://localhost:11434' otherwise.
    """
    if base_url is None:
        host = os.environ.get("OLLAMA_HOST")
        if host:
            if host.startswith(("http://", "https://")):
                base_url = host.rstrip("/")
            else:
                base_url = f"http://{host}"
        else:
            base_url = "http://localhost:11434"
    else:
        base_url = base_url.rstrip("/")

    tags = _http_get_json(f"{base_url}/api/tags")
    if not tags:
        return {"reachable": False, "models": []}

    models: list[str] = []
    for m in tags.get("models", []):
        name = m.get("name")
        if isinstance(name, str):
            models.append(name)

    return {"reachable": True, "models": models}


# Example direct call to the generate endpoint using OLLAMA_HOST.
os.environ["OLLAMA_HOST"] = "172.21.112.1:11434"
host = os.environ.get("OLLAMA_HOST")
print("Using host:", host)

# Normalise host into a full base URL so this also works if you
# export OLLAMA_HOST="http://172.21.112.1:11434".
if host is None:
    raise RuntimeError("OLLAMA_HOST is not set")
if host.startswith(("http://", "https://")):
    base_url = host.rstrip("/")
else:
    base_url = f"http://{host}"

# Don't wait forever on Ollama responses: use connect/read timeouts.
# `timeout` is (connect_timeout_seconds, read_timeout_seconds).
OLLAMA_TIMEOUT_S = (3, 60)

try:
    resp = requests.post(
        f"{base_url}/api/generate",
        json={"model": "llama3.2", "prompt": "Hello from WSL!", "stream": False},
        timeout=OLLAMA_TIMEOUT_S,
    )
    resp.raise_for_status()
    print(resp.json())
except requests.exceptions.Timeout:
    print(f"Ollama request timed out (connect/read): {OLLAMA_TIMEOUT_S}")
except requests.exceptions.RequestException as e:
    print("Ollama request failed:", e)

status = ollama_status()
print("Ollama reachable:", status["reachable"])
print("Models:", status["models"][:10], "..." if len(status["models"]) > 10 else "")

# Compare task inference decisions (rules vs LLM).
rules_gen = DataGenerator(task_inference_mode="rules")
llm_gen = DataGenerator(task_inference_mode="llm", llm_model="llama3.2")

descriptions_to_test = [
    "Extract company, person and location names from short tech news snippets.",
    "Classify customer review sentiment into positive, negative and neutral.",
    "Identify works_for and lives_in relations between people and organizations.",
    "Extract product name, storage and price into a JSON structure.",
    "Extract company names and classify sentiment.",
    "Extract a JSON product record (name, storage, price) and classify the tone as positive, negative, or neutral.",
]

for desc in descriptions_to_test:
    print("\n==== Task description ====\n", desc)
    rules_parsed = rules_gen._infer_tasks_rules(desc)
    print("rules:", asdict(rules_parsed))

    llm_parsed = llm_gen._infer_tasks_via_llm(desc)
    if llm_parsed is None:
        print(
            "llm:   <unavailable (Ollama not running / model missing / invalid JSON)>"
        )
    else:
        print("llm:", asdict(llm_parsed))

In [ ]:
from pprint import pprint

# End-to-end generation using the LLM inference mode.
# Note: if Ollama is unavailable, DataGenerator will fall back to rule-based inference.
llm_generator = DataGenerator(seed=7, task_inference_mode="llm", llm_model="llama3.2")

desc = "From each customer review, extract company names and classify overall sentiment into positive, negative or neutral."
examples = llm_generator.generate(desc, n=3)

print("\n==== LLM-mode generate() output (example 0) ====")
pprint(examples[0])

## Bonus: Fine-tune GLiNER2 on generated data (scaffold)

The following cells show a minimal workflow for fine-tuning `fastino/gliner2-base-v1` on a synthetic sentiment classification task using the generated JSONL data.

This is **scaffold code**: you may need to adjust batch sizes, epochs, and device configuration depending on your hardware.


In [ ]:
import json
import random
from pathlib import Path

from generate import DataGenerator

data_dir = Path("synthetic_data")
data_dir.mkdir(exist_ok=True)

sentiment_description = (
    "Classify customer review sentiment into positive, negative and neutral. "
    "Make the examples diverse across topics."
)


def generate_unique_examples(
    generator: DataGenerator,
    description: str,
    n: int,
) -> list[dict]:
    """Generate n unique examples in a single batch.

    Uniqueness is enforced via a hash of the full example structure. If the
    generator cannot produce enough distinct examples, this loop will keep
    sampling until the space is exhausted.
    """

    seen_keys: set[str] = set()
    examples: list[dict] = []

    while len(examples) < n:
        remaining = n - len(examples)
        # Generate a small oversampled batch to increase the chance of new examples.
        batch = generator.generate(description, n=remaining * 2)
        for ex in batch:
            key = json.dumps(ex, sort_keys=True, ensure_ascii=False)
            if key in seen_keys:
                continue
            seen_keys.add(key)
            examples.append(ex)
            if len(examples) >= n:
                break

    return examples


# Generate a single batch of unique examples, then split 20% into eval.

gen = DataGenerator(seed=123)

total_examples = 100
all_examples = generate_unique_examples(gen, sentiment_description, n=total_examples)

# Deterministic shuffle before splitting.
rng = random.Random(123)
rng.shuffle(all_examples)

eval_size = int(total_examples * 0.2)

eval_examples = all_examples[:eval_size]
train_examples = all_examples[eval_size:]

train_path = data_dir / "train.jsonl"
eval_path = data_dir / "eval.jsonl"

with train_path.open("w", encoding="utf-8") as f:
    for ex in train_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with eval_path.open("w", encoding="utf-8") as f:
    for ex in eval_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

train_path, eval_path

In [ ]:
# Fine-tuning scaffold. This follows the GLiNER2 training docs.
# You may need a GPU and to install the `gliner2` package.

from gliner2 import GLiNER2  # pyright: ignore[reportMissingImports]
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig  # pyright: ignore[reportMissingImports]

model_name = "fastino/gliner2-base-v1"
model = GLiNER2.from_pretrained(model_name)

config = TrainingConfig(
    output_dir="./output_gliner2_sentiment",
    experiment_name="gliner2_sentiment_demo",
    num_epochs=3,
    batch_size=8,
    encoder_lr=5e-5,
    task_lr=5e-5,
)

trainer = GLiNER2Trainer(model, config)
trainer.train(train_data=str(train_path))

### Evaluation metrics

We evaluate on the held-out synthetic set with two kinds of metrics:

- **Classification accuracy**: fraction of examples where the predicted label exactly matches the gold label. We report overall accuracy for both the base and fine-tuned model, and optionally per-label accuracy (useful when labels are imbalanced).
- **Span match (entity-level)** when the eval data contains NER: we treat each `(label, span_text)` pair as one span. A predicted span matches gold if the same text is listed for that label in the gold output. We aggregate **precision** (matched predictions / total predictions), **recall** (matched gold / total gold), and **F1** (harmonic mean of P and R). These are micro-averaged over all entity types.

In [ ]:
# Expanded evaluation: classification accuracy and span-match (entity) metrics.
import json
from collections import defaultdict

from gliner2 import GLiNER2   # pyright: ignore[reportMissingImports]

model_name = "fastino/gliner2-base-v1"
base_model = GLiNER2.from_pretrained(model_name)


def _gold_entities_set(output_entities: dict) -> set[tuple[str, str]]:
    """Flatten output['entities'] to set of (label, span_text) for span-level comparison."""
    out = set()
    for label, spans in (output_entities or {}).items():
        for s in spans:
            if isinstance(s, str) and s.strip():
                out.add((label, s.strip()))
    return out


def _pred_entities_set(pred: dict) -> set[tuple[str, str]]:
    """Flatten GLiNER2 entity prediction to set of (label, span_text)."""
    out = set()
    entities = pred.get("entities") if isinstance(pred, dict) else {}
    for label, spans in (entities or {}).items():
        for s in spans if isinstance(spans, list) else []:
            if isinstance(s, str) and s.strip():
                out.add((label, s.strip()))
    return out


def span_metrics(gold_set: set, pred_set: set) -> dict:
    """Micro P/R/F1 for exact (label, span_text) match."""
    if not gold_set and not pred_set:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}
    matched = len(gold_set & pred_set)
    prec = matched / len(pred_set) if pred_set else 0.0
    rec = matched / len(gold_set) if gold_set else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"precision": prec, "recall": rec, "f1": f1}


# Load eval examples
eval_examples = []
with eval_path.open("r", encoding="utf-8") as f:
    for line in f:
        eval_examples.append(json.loads(line))

# Classification: collect gold labels and task config from first example
has_classification = any(
    ex.get("output", {}).get("classifications") for ex in eval_examples
)
has_entities = any(ex.get("output", {}).get("entities") for ex in eval_examples)

# Task config from first example that has it
cls_task_name = None
cls_labels = None
for ex in eval_examples:
    clss = ex.get("output", {}).get("classifications") or []
    if clss:
        cls_task_name = clss[0].get("task", "classification")
        cls_labels = clss[0].get("labels", ["positive", "negative", "neutral"])
        break

base_cls_correct = 0
ft_cls_correct = 0
per_label_gold: dict[str, int] = defaultdict(int)
per_label_base_correct: dict[str, int] = defaultdict(int)
per_label_ft_correct: dict[str, int] = defaultdict(int)

# Entity span accumulators (for micro P/R/F1)
gold_span_sets: list[set] = []
base_pred_span_sets: list[set] = []
ft_pred_span_sets: list[set] = []

for ex in eval_examples:
    text = ex["input"]
    out = ex.get("output", {})

    if has_classification and out.get("classifications"):
        true_cls = out["classifications"][0]["true_label"][0]
        per_label_gold[true_cls] += 1
        tasks = {cls_task_name: {"labels": cls_labels}}

        base_pred = base_model.classify_text(
            text, tasks=tasks, threshold=0.5,
            format_results=True, include_confidence=False,
        )
        ft_pred = model.classify_text(
            text, tasks=tasks, threshold=0.5,
            format_results=True, include_confidence=False,
        )
        base_cls = base_pred.get(cls_task_name)
        ft_cls = ft_pred.get(cls_task_name)

        if base_cls == true_cls:
            base_cls_correct += 1
            per_label_base_correct[true_cls] += 1
        if ft_cls == true_cls:
            ft_cls_correct += 1
            per_label_ft_correct[true_cls] += 1

    if has_entities and out.get("entities"):
        gold_span_sets.append(_gold_entities_set(out["entities"]))
        entity_labels = list(out["entities"].keys())
        base_ent = base_model.extract_entities(text, entity_labels)
        ft_ent = model.extract_entities(text, entity_labels)
        base_pred_span_sets.append(_pred_entities_set(base_ent))
        ft_pred_span_sets.append(_pred_entities_set(ft_ent))

# Build results
n_total = len(eval_examples)
n_cls = sum(per_label_gold.values()) if has_classification else 0

results = {}

if has_classification and n_cls > 0:
    results["classification"] = {
        "num_examples": n_cls,
        "base_accuracy": base_cls_correct / n_cls,
        "fine_tuned_accuracy": ft_cls_correct / n_cls,
    }
    per_label_accuracy = {}
    for label in per_label_gold:
        g = per_label_gold[label]
        per_label_accuracy[label] = {
            "count": g,
            "base_accuracy": per_label_base_correct[label] / max(1, g),
            "fine_tuned_accuracy": per_label_ft_correct[label] / max(1, g),
        }
    results["classification"]["per_label"] = per_label_accuracy

if has_entities and gold_span_sets:
    all_gold = set()
    all_base = set()
    all_ft = set()
    for a, b, c in zip(gold_span_sets, base_pred_span_sets, ft_pred_span_sets):
        all_gold |= a
        all_base |= b
        all_ft |= c
    results["span_match"] = {
        "num_spans_gold": len(all_gold),
        "base": span_metrics(all_gold, all_base),
        "fine_tuned": span_metrics(all_gold, all_ft),
    }

results

In [ ]:
# Summarise and compare metrics
if "classification" in results:
    c = results["classification"]
    print("Classification (accuracy)")
    print(f"  Base:       {c['base_accuracy']:.2%}  Fine-tuned: {c['fine_tuned_accuracy']:.2%}")
    if "per_label" in c:
        for label, stats in c["per_label"].items():
            print(f"  [{label}] n={stats['count']}  base={stats['base_accuracy']:.2%}  ft={stats['fine_tuned_accuracy']:.2%}")
if "span_match" in results:
    s = results["span_match"]
    print("Span match (entity-level micro P/R/F1)")
    print(f"  Base:       P={s['base']['precision']:.3f} R={s['base']['recall']:.3f} F1={s['base']['f1']:.3f}")
    print(f"  Fine-tuned: P={s['fine_tuned']['precision']:.3f} R={s['fine_tuned']['recall']:.3f} F1={s['fine_tuned']['f1']:.3f}")